# Can we predict late deliveries?
Target: `late_delivery`. Prediction is made at dispatch, for future consignments, so the split is temporal.

Decisions up front (checklist.md):
- Each consignment appears once, so no entity grouping is needed. The test set is the **latest 20% of rows by `dispatched_on`**.
- `consignment` is an identifier: dropped.
- `claim_paid_eur` is a compensation paid after the delivery outcome, so it is not known at dispatch: dropped (leakage).
- Features from `dispatched_on` (hour, weekday) are known at dispatch and are kept.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
df['hour'] = df.dispatched_on.dt.hour
df['weekday'] = df.dispatched_on.dt.weekday
print(df.shape, 'duplicate ids:', df.consignment.duplicated().sum(), 'missing:', int(df.isna().sum().sum()))
print('late rate:', df.late_delivery.mean().round(3))

## 1. Test set set aside (latest 20% of rows)

In [ ]:
n_test = int(len(df) * 0.2)
train, test = df.iloc[:-n_test].copy(), df.iloc[-n_test:].copy()
print('train', train.dispatched_on.min(), '->', train.dispatched_on.max(), len(train))
print('test ', test.dispatched_on.min(), '->', test.dispatched_on.max(), len(test))
FEATURES = ['carrier','distance_km','weight_kg','parcels','hour','weekday']
NUM = ['distance_km','weight_kg','parcels','hour','weekday']
X_tr, y_tr = train[FEATURES], train.late_delivery
# test set is not touched again until step 6

## 2-4. Candidate models (named before any scoring) and CV structure
Candidates:
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg_C0.1`, `logreg_C1`, `logreg_C10`: logistic regression, one-hot carrier + standardised numerics.
3. `hgb_shallow`, `hgb_deep`: histogram gradient boosting (depth 3 / lr 0.05, and depth 6 / lr 0.1).

CV: expanding-window `TimeSeriesSplit(5)` inside the training rows only (earlier rows train, later rows validate). All preprocessing lives inside the Pipeline, so it is fit per fold. Selection metric: mean ROC AUC across folds (threshold-free). The decision threshold is then chosen on pooled out-of-fold predictions to maximise F1.

In [ ]:
def prep():
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['carrier']),
                              ('num', StandardScaler(), NUM)])
def hgb_prep():
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['carrier'])], remainder='passthrough')

CANDIDATES = {
    'baseline':    Pipeline([('m', DummyClassifier(strategy='prior'))]),
    'logreg_C0.1': Pipeline([('p', prep()), ('m', LogisticRegression(C=0.1, max_iter=1000))]),
    'logreg_C1':   Pipeline([('p', prep()), ('m', LogisticRegression(C=1, max_iter=1000))]),
    'logreg_C10':  Pipeline([('p', prep()), ('m', LogisticRegression(C=10, max_iter=1000))]),
    'hgb_shallow': Pipeline([('p', hgb_prep()), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))]),
    'hgb_deep':    Pipeline([('p', hgb_prep()), ('m', HistGradientBoostingClassifier(max_depth=6, learning_rate=0.1, max_iter=200, random_state=0))]),
}
cv = TimeSeriesSplit(n_splits=5)
oof = {}; rows = []
for name, est in CANDIDATES.items():
    pred = pd.Series(np.nan, index=X_tr.index); aucs=[]; aps=[]
    for a, b in cv.split(X_tr):
        m = clone(est).fit(X_tr.iloc[a], y_tr.iloc[a])
        p = m.predict_proba(X_tr.iloc[b])[:, 1]
        pred.iloc[b] = p
        aucs.append(roc_auc_score(y_tr.iloc[b], p)); aps.append(average_precision_score(y_tr.iloc[b], p))
    oof[name] = pred
    rows.append((name, np.mean(aucs), np.std(aucs), np.mean(aps)))
cv_table = pd.DataFrame(rows, columns=['model','cv_auc','cv_auc_sd','cv_ap']).set_index('model').round(4)
cv_table

## 5. Choice on CV only (made before scoring the test set)

In [ ]:
best = cv_table.drop(index='baseline').cv_auc.idxmax()
mask = oof[best].notna()
grid = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y_tr[mask], oof[best][mask] >= t) for t in grid]
THRESH = float(grid[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(THRESH, 2), '| CV F1 at threshold:', round(max(f1s), 3))
print('CV F1 of always-predict-late:', round(f1_score(y_tr[mask], np.ones(mask.sum())), 3))

**Choice recorded:** the model and threshold printed above (selected by CV AUC and OOF-F1 respectively), with features `carrier, distance_km, weight_kg, parcels, hour, weekday`. Nothing below changes them.

## 6. Refit on all training rows; score the test set once (headline)

In [ ]:
final = clone(CANDIDATES[best]).fit(X_tr, y_tr)
base = clone(CANDIDATES['baseline']).fit(X_tr, y_tr)
X_te, y_te = test[FEATURES], test.late_delivery
def report(m, thr=None):
    p = m.predict_proba(X_te)[:, 1]
    out = {'auc': roc_auc_score(y_te, p), 'ap': average_precision_score(y_te, p)}
    if thr is not None:
        yh = p >= thr
        out |= {'precision': precision_score(y_te, yh, zero_division=0), 'recall': recall_score(y_te, yh), 'f1': f1_score(y_te, yh)}
    return out
res = pd.DataFrame({best: report(final, THRESH), 'baseline': report(base)}).T.round(4)
print('test late rate:', y_te.mean().round(3))
res

## Interpretation
Results from the run (random_state fixed): CV picked `logreg_C0.1` (CV AUC 0.722 vs 0.5 baseline) with threshold 0.24. On the held-out latest 20% it scored **AUC 0.70, AP 0.069** (baseline AP 0.022), precision 0.04, recall 0.71, F1 0.08.

So the features carry a modest ranking signal that survives into the future period, but the threshold-based decisions do not transfer: the late rate drifts strongly over time (~30% in much of 2024-25, then falling to ~1-5% from Oct 2025; the test period is 2.2% late). A threshold tuned at a ~25% base rate flags far too many consignments when lateness is rare, hence the low precision. Distinguishing a real change in the process from a data-recording artefact needs domain input. Caveats: single temporal holdout; only about 35 late cases in the test set, so estimates are noisy; `claim_paid_eur` excluded because it is only known after the outcome.